In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_by_grid_with_auto_class_mapping(shapefile_path, grid_path, class_field, output_dir, resolution=10, nodata=255):
    # 读取数据
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统统一为投影坐标系（UTM 51N）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 清理无效几何
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    # 检查字段是否存在
    if class_field not in gdf.columns:
        raise ValueError(f"字段 {class_field} 不存在于 shapefile 中")

    # 获取字段的唯一值列表，并生成映射字典（类别字符串->数字ID，从1开始，0 留作背景）
    unique_classes = sorted(gdf[class_field].dropna().unique())
    class_to_id = {cls: idx+1 for idx, cls in enumerate(unique_classes)}
    print(f"自动生成类别映射（{class_field}）：{class_to_id}")

    # 用映射替换原字段，未包含的赋值0
    gdf['class_val'] = gdf[class_field].map(class_to_id).fillna(0).astype(int)

    os.makedirs(output_dir, exist_ok=True)

    for idx, cell in grid.iterrows():
        grid_geom = cell.geometry

        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=nodata,
            dtype='uint8'
        )
        tile_id = cell["ID"]
        out_path = os.path.join(output_dir, f"street_tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=nodata
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 使用示例
rasterize_by_grid_with_auto_class_mapping(
    shapefile_path=r"D:\shanghai workshop\vector\Shanghai street_ver3\Shanghai street_ver3.shp",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    class_field="Type",
    output_dir=r"D:\shanghai workshop\raster\street_tiles",
    resolution=1,
    nodata=255
)

自动生成类别映射（Type）：{'中心城区主要道路': 1, '中心城区道路': 2, '主干道路': 3, '公路': 4, '快速路': 5, '快速路匝道': 6, '支路': 7, '郊县中心城区主干道路': 8, '高速': 9, '高速公路匝道': 10}
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_194.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_195.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_196.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_198.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_199.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_539.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_540.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_541.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_542.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_543.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_544.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_545.tif
✅ 输出: D:\shanghai workshop\raster\street_tiles\street_tile_54